# BanglaHalluEval: Distractor Answer Generation (Full 1000)

This notebook generates hallucinated (distractor) answers for the **BanglaHalluEval 1000-row dataset** using OpenAI's API. 

### Workflow:
1. **Install Dependencies**: Run the first cell to install the `openai` library.
2. **Setup API Key**: Enter your OpenAI API Key in the configuration cell.
3. **Upload Dataset**: Upload the `banglahallueval_qa_1000.csv` file when prompted.
4. **Run Generation**: Execute the generation loop. It will process each row through 4 distractor patterns.
5. **Download Results**: The final results file will be automatically downloaded.

In [ ]:
# @title 1. Install Dependencies
!pip install openai

In [ ]:
# @title 2. Configuration & API Setup
import csv
import os
import time
import io
from pathlib import Path
from google.colab import files, userdata
from openai import OpenAI

# @markdown Enter your OpenAI API Key below:
try:
    # Check if key is in Colab Secrets
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
except:
    OPENAI_API_KEY = "" # @param {type:"string"}

if not OPENAI_API_KEY:
    print("ERROR: Please enter your OpenAI API Key in the field above.")
else:
    print("API Key configured.")

# @markdown Default Model for this task:
MODEL_NAME = "gpt-5.4" # @param {type:"string"}

client = OpenAI(api_key=OPENAI_API_KEY)

In [ ]:
# @title 3. Upload Dataset (`banglahallueval_qa_1000.csv`)
print("Please upload 'banglahallueval_qa_1000.csv'")
uploaded = files.upload()
INPUT_FILENAME = list(uploaded.keys())[0]
print(f"Successfully uploaded: {INPUT_FILENAME}")

In [ ]:
# @title 4. Utility Functions

def load_rows(csv_data: bytes) -> list[dict]:
    f = io.StringIO(csv_data.decode("utf-8"))
    return list(csv.DictReader(f))

def write_rows(filename: str, rows: list[dict], fieldnames: list[str]) -> None:
    # Using utf-8-sig for Excel compatibility with Bengali text
    with open(filename, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

def build_prompt(template: str, context: str, question: str, right_answer: str) -> str:
    return template.replace("<insert the related knowledge/context>", context).replace(
        "<insert the question>", question
    ).replace("<insert the right answer to the question>", right_answer)

def normalize_answer(text: str) -> str:
    cleaned = text.strip()
    if not cleaned: return cleaned
    
    marker = "#Hallucinated Answer#"
    if marker in cleaned:
        cleaned = cleaned.split(marker, 1)[-1].strip()
    
    lines = [line.strip() for line in cleaned.splitlines() if line.strip()]
    return lines[0] if lines else cleaned

def request_hallucination(client, model, prompt):
    # Note: Using the specific signature from the original script
    request_kwargs = {
        "model": model,
        "input": [{"role": "user", "content": prompt}],
        "max_output_tokens": 64,
        "temperature": 0.7,
    }
    response = client.responses.create(**request_kwargs)
    return response.output_text

In [ ]:
# @title 5. Define Distractor Patterns

PROMPT_1_FACTUAL = (
    "I want you act as a hallucination answer generator. The answer should be given in BANGLA. Given a "
    "question, right answer, and related knowledge, your objective is to write a hallucinated answer that "
    "sounds plausible but is factually incorrect. You SHOULD write the hallucinated answer using the "
    "following method: You are trying to answer a question but there is a factual contradiction between the "
    "answer and the knowledge. You can fabricate some information that does not exist in the provided "
    "knowledge.\n"
    "#Knowledge#: \"উইলিয়াম আব্রাহাম সাইমন ঔডারল্যান্ড (Dutch: Wiliam Ouderland) (জন্ম: ৬ ডিসেম্বর, ১৯১৭ — "
    "মৃত্যু: ১৮ই মে, ২০০১) ছিলেন একজন ওলন্দাজ-অস্ট্রেলীয় সামরিক কমান্ডো অফিসার। তিনি দ্বিতীয় বিশ্বযুদ্ধে "
    "সক্রিয়ভাবে অংশগ্রহণ করেন। বাংলাদেশ সরকার তাঁকে বাংলাদেশের চতুর্থ সর্বোচ্চ সামরিক খেতাব বীর প্রতীক প্রদান করে।\"\n"
    "#Question#: \"ওলন্দাজ-অস্ট্রেলীয় সামরিক কমান্ডো অফিসার উইলিয়াম আব্রাহাম সাইমন ঔডারল্যান্ড কবে জন্মগ্রহণ করেন ?\"\n"
    "#Right Answer#: \"৬ ডিসেম্বর, ১৯১৭\"\n"
    "#Hallucinated Answer#: \"৬ নভেম্বর, ১৯১৬\"\n"
    "You should try your best to make the answer become hallucinated. #Hallucinated Answer# can only have "
    "about 5 more words than #Right Answer#.\n"
    "#Knowledge#: <insert the related knowledge/context>\n"
    "#Question#: <insert the question>\n"
    "#Right Answer#: <insert the right answer to the question>\n"
    "#Hallucinated Answer#: Generate"
)

PROMPT_2_COMPREHENSION = (
    "I want you act as a hallucination answer generator. The answer should be given in BANGLA. Given a "
    "question, right answer, and related knowledge, your objective is to write a hallucinated answer that "
    "sounds plausible but is factually incorrect. You SHOULD write the hallucinated answer using the "
    "following method: You are trying to answer a question but you misunderstand the question context and "
    "intention.\n"
    "#Knowledge#: \"1927-28 সালে ঢাকায় প্রথম চলচ্চিত্র নির্মিত হয়। নওয়াব পরিবারের কয়েকজন তরুণ সংস্কৃতিসেবী নির্মাণ করেন চলচ্চিত্র সুকুমারী।\"\n"
    "#Question#: \"স্বাধীন বাংলাদেশের প্রথম চলচ্চিত্রটির নাম কী ?\"\n"
    "#Right Answer#: \"সুকুমারী\"\n"
    "#Hallucinated Answer#: \"জাহির রাইহান\"\n"
    "You should try your best to make the answer become hallucinated. #Hallucinated Answer# can only have "
    "about 5 more words than #Right Answer#.\n"
    "#Knowledge#: <insert the related knowledge/context>\n"
    "#Question#: <insert the question>\n"
    "#Right Answer#: <insert the right answer to the question>\n"
    "#Hallucinated Answer#: Generate"
)

PROMPT_3_SPECIFICITY = (
    "I want you act as a hallucination answer generator. The answer should be given in BANGLA. Given a "
    "question, right answer, and related knowledge, your objective is to write a hallucinated answer that "
    "sounds plausible but is factually incorrect. You SHOULD write the hallucinated answer using the "
    "following method: You are trying to answer a question but the answer is too general or too specific "
    "to answer the question at an appropriate level of specificity.\n"
    "#Knowledge#: \"খুলনা প্রকৌশল ও প্রযুক্তি বিশ্ববিদ্যালয় (কুয়েট) বাংলাদেশের একটি অন্যতম সরকারি প্রকৌশল বিশ্ববিদ্যালয়। এখানে প্রায় ৬ হাজার জন ছাত্রছাত্রী পড়াশোনা করছে।\"\n"
    "#Question#: \"বর্তমানে খুলনা প্রকৌশল ও প্রযুক্তি বিশ্ববিদ্যালয়ের মোট ছাত্রছাত্রীর সংখ্যা কত ?\"\n"
    "#Right Answer#: \"প্রায় ৬ হাজার\"\n"
    "#Hallucinated Answer#: \"অজানা\"\n"
    "You should try your best to make the answer become hallucinated. #Hallucinated Answer# can only have "
    "about 5 more words than #Right Answer#.\n"
    "#Knowledge#: <insert the related knowledge/context>\n"
    "#Question#: <insert the question>\n"
    "#Right Answer#: <insert the right answer to the question>\n"
    "#Hallucinated Answer#: Generate"
)

PROMPT_4_INFERENCE = (
    "I want you act as a hallucination answer generator. The answer should be given in BANGLA. Given a "
    "question, right answer, and related knowledge, your objective is to write a hallucinated answer that "
    "sounds plausible but is factually incorrect. You SHOULD write the hallucinated answer using the "
    "following method: You are trying to answer a question but the answer cannot be inferred from the "
    "knowledge. You can incorrectly reason with the knowledge to arrive at a hallucinated answer.\n"
    "#Knowledge#: \"ঢাকা দক্ষিণ এশিয়ার রাষ্ট্র বাংলাদেশের রাজধানী ও বৃহত্তম শহর। ভৌগোলিকভাবে এটি বাংলাদেশের মধ্যভাগে বুড়িগঙ্গা নদীর উত্তর তীরে অবস্থিত।\"\n"
    "#Question#: \"ঢাকার মোট আয়তন কত ?\"\n"
    "#Right Answer#: \"১৩৪ বর্গমাইল\"\n"
    "#Hallucinated Answer#: \"২০ মিলিয়ন জনসংখ্যা\"\n"
    "You should try your best to make the answer become hallucinated. #Hallucinated Answer# can only have "
    "about 5 more words than #Right Answer#.\n"
    "#Knowledge#: <insert the related knowledge/context>\n"
    "#Question#: <insert the question>\n"
    "#Right Answer#: <insert the right answer to the question>\n"
    "#Hallucinated Answer#: Generate"
)

PROMPT_MAP = {
    "factualness": PROMPT_1_FACTUAL,
    "comprehension": PROMPT_2_COMPREHENSION,
    "specificity": PROMPT_3_SPECIFICITY,
    "inference": PROMPT_4_INFERENCE,
}

In [ ]:
# @title 6. Run Generation Loop

rows = load_rows(uploaded[INPUT_FILENAME])
output_rows = []
output_filename = "hallucinated_answers_generation_full_1000.csv"

total = len(rows) * len(PROMPT_MAP)
count = 0

print(f"Processing {len(rows)} source rows ({total} tasks)...")

for i, row in enumerate(rows):
    source_id = row.get("id", "").strip()
    context = row.get("context", "").strip()
    question = row.get("question", "").strip()
    right_answer = row.get("correct_answer", "").strip()

    for pattern, template in PROMPT_MAP.items():
        prompt = build_prompt(template, context, question, right_answer)
        try:
            raw = request_hallucination(client, MODEL_NAME, prompt)
            hallucinated = normalize_answer(raw)
        except Exception as e:
            print(f"Error on {source_id} {pattern}: {e}")
            hallucinated = "ERROR"
        
        output_rows.append({
            "id": f"{source_id}::{pattern}",
            "source_id": source_id,
            "pattern": pattern,
            "context": context,
            "question": question,
            "right_answer": right_answer,
            "hallucinated_answer": hallucinated
        })
        
        count += 1
        if count % 10 == 0:
            print(f"Progress: {count}/{total} (Row {i+1}/{len(rows)})")
        
        # Small delay to avoid hitting rate limits too hard
        time.sleep(0.1)

fieldnames = ["id", "source_id", "pattern", "context", "question", "right_answer", "hallucinated_answer"]
write_rows(output_filename, output_rows, fieldnames)
print(f"\nCompleted! Processed {len(output_rows)} total distractor entries.")

In [ ]:
# @title 7. Download Results
files.download(output_filename)